# Aggregating, Reshaping and Combining DataFrames

## Introduction to Aggregation, Reshaping and Combining

There are a number of functions:
- **Aggregation** calculates the summary of each group.
- **Reshaping** rearanges the representation but doesn`t nessesarily calculate a summary.
- **Combining** stacks records/matches them using a key; it brings seperate records together.

**Common Aggregation Functions**
- ***sum():*** Calculates total of numeric values.
- ***mean():*** Calculates arithmetic average.
- ***median():*** Middle value after sorting.
- ***min() & max():*** Smallest and largest value.
- ***count():*** Counts present values in a chosen field.
- ***nunique():*** Number of distinct values.
- ***size():*** Numbers of rows in a column.

## Process for aggregating, reshaping and combining dataframes
- Choose the grouping columns and the value to surmarise.
- Select an aggregation function that matches the question and units.
- Report the missing values/observation counts alongside summaries.
- Reshape only when the new arrangement helps compare values.
- For a merge, identify the common key and expected relationship.
- Validate output row counts, unmatched keys and totals before reporting.



In [1]:
import pandas as pd
clinic_visits = pd.read_csv("clinic_visits_cleaned.csv")
clinic_visits



FileNotFoundError: [Errno 2] No such file or directory: 'clinic_visits_cleaned.csv'

In [ ]:
clinic_visits_copy = clinic_visits.copy()
clinic_visits_copy.head()

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes
0,V001,P01,C01,2026-01-05,24.0,consultation,1200.0,30.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800.0,45.0
2,V004,P04,C03,2026-01-06,42.0,Pharmacy,500.0,15.0
3,V005,P05,C02,2026-02-03,29.0,consultation,1200.0,35.0
4,V006,P01,C02,2026-02-03,24.0,Lab,800.0,35.0


## Questions for aggregation summaries
1. Check for the aggregation summaries.
2. Group by any cartegory.
3. Group by 2 columns.

In [ ]:
clinic_visits_copy.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   visit_id      10 non-null     str    
 1   patient_id    10 non-null     str    
 2   clinic_id     10 non-null     str    
 3   visit_date    9 non-null      str    
 4   age           10 non-null     float64
 5   service       10 non-null     str    
 6   fee_kes       9 non-null      float64
 7   wait_minutes  10 non-null     float64
dtypes: float64(3), str(5)
memory usage: 772.0 bytes


In [ ]:
# Question 1: Calculate average wait time(mean), fees(mean and sum), unique ids(unique)
clinic_visits_copy.describe(include='all')

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes
count,10,10,10,9,10.000000,10,9.000000,10.000000
unique,10,9,3,6,NaN,7,NaN,NaN
top,V001,P01,C02,2026-01-05,NaN,Lab,NaN,NaN
freq,1,2,4,2,NaN,3,NaN,NaN
mean,NaN,NaN,NaN,NaN,37.800000,NaN,911.111111,35.000000
std,NaN,NaN,NaN,NaN,11.659998,NaN,297.676185,15.092309
min,NaN,NaN,NaN,NaN,24.000000,NaN,500.000000,10.000000
25%,NaN,NaN,NaN,NaN,29.500000,NaN,800.000000,30.000000
50%,NaN,NaN,NaN,NaN,36.500000,NaN,800.000000,35.000000
75%,NaN,NaN,NaN,NaN,44.250000,NaN,1200.000000,43.750000


In [ ]:
# Question 2: Group by any category. 
# Get the fee summary for each clinic.
clinic_visits_copy.groupby('clinic_id')['fee_kes'].describe()

,count,mean,std,min,25%,50%,75%,max
clinic_id,,,,,,,,
C01,2.0,1000.000000,282.842712,800.0,900.0,1000.0,1100.0,1200.0
C02,4.0,1000.000000,230.940108,800.0,800.0,1000.0,1200.0,1200.0
C03,3.0,733.333333,404.145188,500.0,500.0,500.0,850.0,1200.0


In [ ]:
# Question 3: Group by 2 columns
# Get the fee summary for all patients per clinic.
clinic_visits_copy.groupby(['clinic_id', 'patient_id'])['fee_kes'].describe()

count    mean         std     min     25%     50%  \
clinic_id patient_id                                                      
C01       P01           2.0  1000.0  282.842712   800.0   900.0  1000.0   
          P08           0.0     NaN         NaN     NaN     NaN     NaN   
C02       P02           1.0   800.0         NaN   800.0   800.0   800.0   
          P05           1.0  1200.0         NaN  1200.0  1200.0  1200.0   
          P07           1.0  1200.0         NaN  1200.0  1200.0  1200.0   
          P10           1.0   800.0         NaN   800.0   800.0   800.0   
C03       P04           1.0   500.0         NaN   500.0   500.0   500.0   
          P09           1.0   500.0         NaN   500.0   500.0   500.0   
          P11           1.0  1200.0         NaN  1200.0  1200.0  1200.0   

                         75%     max  
clinic_id patient_id                  
C01       P01         1100.0  1200.0  
          P08            NaN     NaN  
C02       P02          800.0   800.0  
          P05         1200.0  1200.0  
          P07         1200.0  1200.0  
          P10          800.0   800.0  
C03       P04          500.0   500.0  
          P09          500.0   500.0  
          P11         1200.0  1200.0

## Questions for reshaping 
1. Explain and create a pivot table.
2. Return a wide table to longform using melt.
- Wide form spreads categories across columns e.g. service_grid.
- Longform stores category manes in one column and stores values in another 

 

In [ ]:
clinic_pivot = clinic_visits_copy.pivot_table(
    values='fee_kes',
    index= 'patient_id',
    columns= 'clinic_id',
    aggfunc='sum')
print(clinic_pivot)

clinic_id      C01     C02     C03
patient_id                        
P01         1200.0   800.0     NaN
P02            NaN   800.0     NaN
P04            NaN     NaN   500.0
P05            NaN  1200.0     NaN
P07            NaN  1200.0     NaN
P08            0.0     NaN     NaN
P09            NaN     NaN   500.0
P10            NaN   800.0     NaN
P11            NaN     NaN  1200.0


In [ ]:
longform_clinic_visit = clinic_pivot.reset_index(names='patient_id').melt(id_vars=['patient_id'], var_name='clinic_id', value_name= 'fee_kes')
print(longform_clinic_visit)

   patient_id clinic_id  fee_kes
0         P01       C01   1200.0
1         P02       C01      NaN
2         P04       C01      NaN
3         P05       C01      NaN
4         P07       C01      NaN
5         P08       C01      0.0
6         P09       C01      NaN
7         P10       C01      NaN
8         P11       C01      NaN
9         P01       C02    800.0
10        P02       C02    800.0
11        P04       C02      NaN
12        P05       C02   1200.0
13        P07       C02   1200.0
14        P08       C02      NaN
15        P09       C02      NaN
16        P10       C02    800.0
17        P11       C02      NaN
18        P01       C03      NaN
19        P02       C03      NaN
20        P04       C03    500.0
21        P05       C03      NaN
22        P07       C03      NaN
23        P08       C03      NaN
24        P09       C03    500.0
25        P10       C03      NaN
26        P11       C03   1200.0


## Questions for Combining dataframes 
1. Merge tables using a common key(use the clinc_visits_raw.csv and clinics.csv)
2. Display clinics with no visits.
3. Concatinate the match.csv file to the most appropriate csv file.
4. Extract date components and add them as new columns.
 

In [ ]:
clinic_visits_raw = pd.read_csv('clinic_visits_raw.csv')
clinics_names = pd.read_csv('clinics.csv')
merged_clinic_data = pd.merge(clinic_visits_raw, clinics_names, on='clinic_id', how='inner')


In [ ]:
clinic_visits_raw.head()

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0
2,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0
3,V004,P04,C03,2026-01-06,42.0,Pharmacy,500,15.0
4,V005,P05,C02,2026-02-03,29.0,consultation,1200,NaN


In [ ]:
clinics_names.head()

,clinic_id,clinic_name,county
0,C01,Central Clinic,Nairobi
1,C02,Lake Clinic,Kisumu
2,C03,Coast Clinic,Mombasa
3,C04,Highlands Clinic,Nakuru


In [ ]:
merged_clinic_data

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes,clinic_name,county
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,Central Clinic,Nairobi
1,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,Lake Clinic,Kisumu
2,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,Central Clinic,Nairobi
3,V004,P04,C03,2026-01-06,42.0,Pharmacy,500,15.0,Coast Clinic,Mombasa
4,V005,P05,C02,2026-02-03,29.0,consultation,1200,NaN,Lake Clinic,Kisumu
5,V006,P01,C01,2026-02-03,24.0,Lab,800,35.0,Central Clinic,Nairobi
6,V007,P06,C03,2026-02-04,-5.0,Pharmacy,500,20.0,Coast Clinic,Mombasa
7,V008,P07,C02,2026-02-04,31.0,Consultation,1200,40.0,Lake Clinic,Kisumu
8,V009,P08,C01,2026-02-05,50.0,Lab,unknown,50.0,Central Clinic,Nairobi
9,V010,P09,C03,2026-02-05,38.0,Pharmacy,500,10.0,Coast Clinic,Mombasa


In [ ]:
merged_clinic_data_1 = pd.merge(clinic_visits_raw, clinics_names, on='clinic_id', how='outer')

In [ ]:
# To do
# Look up on inner, outer, left & right joins in pandas.
merged_clinic_data_1

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes,clinic_name,county
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,Central Clinic,Nairobi
1,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,Central Clinic,Nairobi
2,V006,P01,C01,2026-02-03,24.0,Lab,800,35.0,Central Clinic,Nairobi
3,V009,P08,C01,2026-02-05,50.0,Lab,unknown,50.0,Central Clinic,Nairobi
4,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,Lake Clinic,Kisumu
5,V005,P05,C02,2026-02-03,29.0,consultation,1200,NaN,Lake Clinic,Kisumu
6,V008,P07,C02,2026-02-04,31.0,Consultation,1200,40.0,Lake Clinic,Kisumu
7,V011,P10,C02,not-recorded,45.0,Lab,800,30.0,Lake Clinic,Kisumu
8,V004,P04,C03,2026-01-06,42.0,Pharmacy,500,15.0,Coast Clinic,Mombasa
9,V007,P06,C03,2026-02-04,-5.0,Pharmacy,500,20.0,Coast Clinic,Mombasa


In [ ]:
clinics_names.set_index('clinic_id', inplace=True)

In [ ]:
merged_clinic_data_2 = clinic_visits_raw.join(clinics_names, on='clinic_id', how='inner')
merged_clinic_data_2

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes,clinic_name,county
0,V001,P01,C01,2026-01-05,24.0,consultation,1200,30.0,Central Clinic,Nairobi
1,V002,P02,C02,2026-01-05,35.0,LAB,800,45.0,Lake Clinic,Kisumu
2,V003,P03,C01,2026-01-06,NaN,Consultation,1200,25.0,Central Clinic,Nairobi
3,V004,P04,C03,2026-01-06,42.0,Pharmacy,500,15.0,Coast Clinic,Mombasa
4,V005,P05,C02,2026-02-03,29.0,consultation,1200,NaN,Lake Clinic,Kisumu
5,V006,P01,C01,2026-02-03,24.0,Lab,800,35.0,Central Clinic,Nairobi
6,V007,P06,C03,2026-02-04,-5.0,Pharmacy,500,20.0,Coast Clinic,Mombasa
7,V008,P07,C02,2026-02-04,31.0,Consultation,1200,40.0,Lake Clinic,Kisumu
8,V009,P08,C01,2026-02-05,50.0,Lab,unknown,50.0,Central Clinic,Nairobi
9,V010,P09,C03,2026-02-05,38.0,Pharmacy,500,10.0,Coast Clinic,Mombasa


In [ ]:
march_visits = pd.read_csv('march_visits.csv')
march_visits

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes
0,V013,P12,C01,2026-03-01,33,Lab,800,25
1,V014,P13,C02,2026-03-02,27,Pharmacy,500,15


In [ ]:

merged_clinic_data_3 = pd.concat([clinic_visits_copy, march_visits], ignore_index=True)
merged_clinic_data_3

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes
0,V001,P01,C01,2026-01-05,24.0,consultation,1200.0,30.0
1,V002,P02,C02,2026-01-05,35.0,LAB,800.0,45.0
2,V004,P04,C03,2026-01-06,42.0,Pharmacy,500.0,15.0
3,V005,P05,C02,2026-02-03,29.0,consultation,1200.0,35.0
4,V006,P01,C02,2026-02-03,24.0,Lab,800.0,35.0
5,V008,P07,C02,2026-02-04,31.0,Consultation,1200.0,40.0
6,V009,P08,C01,2026-02-05,50.0,Lab,NaN,50.0
7,V010,P09,C03,2026-02-05,38.0,Pharmacy,500.0,10.0
8,V011,P10,C02,NaN,45.0,Lab,800.0,30.0
9,V012,P11,C03,2026-02-06,60.0,Consultation,1200.0,60.0


In [ ]:
merged_clinic_data_3['visit_date'] = pd.to_datetime(merged_clinic_data_3['visit_date'])
merged_clinic_data_3['year'] = merged_clinic_data_3['visit_date'].dt.year
merged_clinic_data_3['month'] = merged_clinic_data_3['visit_date'].dt.month_name()
merged_clinic_data_3['day_of_the_week'] = merged_clinic_data_3['visit_date'].dt.day_name()
merged_clinic_data_3

,visit_id,patient_id,clinic_id,visit_date,age,service,fee_kes,wait_minutes,year,month,day_of_the_week
0,V001,P01,C01,2026-01-05,24.0,consultation,1200.0,30.0,2026.0,January,Monday
1,V002,P02,C02,2026-01-05,35.0,LAB,800.0,45.0,2026.0,January,Monday
2,V004,P04,C03,2026-01-06,42.0,Pharmacy,500.0,15.0,2026.0,January,Tuesday
3,V005,P05,C02,2026-02-03,29.0,consultation,1200.0,35.0,2026.0,February,Tuesday
4,V006,P01,C02,2026-02-03,24.0,Lab,800.0,35.0,2026.0,February,Tuesday
5,V008,P07,C02,2026-02-04,31.0,Consultation,1200.0,40.0,2026.0,February,Wednesday
6,V009,P08,C01,2026-02-05,50.0,Lab,NaN,50.0,2026.0,February,Thursday
7,V010,P09,C03,2026-02-05,38.0,Pharmacy,500.0,10.0,2026.0,February,Thursday
8,V011,P10,C02,NaT,45.0,Lab,800.0,30.0,NaN,NaN,NaN
9,V012,P11,C03,2026-02-06,60.0,Consultation,1200.0,60.0,2026.0,February,Friday
